In [40]:
import pandas as pd,torch,os
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer,AutoModelForSequenceClassification,TrainingArguments,Trainer,DataCollatorWithPadding
from torch.nn.functional import softmax

In [16]:
train_df = pd.read_csv("train.csv")

Setup (Run Before Attempting Questions) :

Use the following two fine-tuned sequence classification checkpoints:

DeBERTa: microsoft/deberta-v3-small (fine-tuned checkpoint)

RoBERTa: roberta-base (fine-tuned checkpoint)

Label Mapping
The models output logits for five labels corresponding to the answer options:

Label ID - Option

0 - A

1 - B

2 - C

3 - D

4 - E

In [36]:
TEXT_COL="prompt"
LABEL_COL="answer"

device="cuda" if torch.cuda.is_available() else "cpu"
print("Device:",device)

df=pd.read_csv("train.csv")
df=df[[TEXT_COL,LABEL_COL]].dropna()
df[LABEL_COL]=df[LABEL_COL].astype(str).str.strip().str.upper().str[0].map({"A":0,"B":1,"C":2,"D":3,"E":4})
df=df.dropna()
dataset=Dataset.from_pandas(df)

def train_model(model_name,save_name,base):
    tokenizer=AutoTokenizer.from_pretrained(model_name)
    def tokenize(x):
        return tokenizer(x[TEXT_COL],truncation=True,max_length=256)
    ds=dataset.map(tokenize,batched=True)
    ds=ds.rename_column(LABEL_COL,"labels")
    remove=[TEXT_COL]
    if "__index_level_0__" in ds.column_names:
        remove.append("__index_level_0__")
    ds=ds.remove_columns(remove)

    model=AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=5)

    args=TrainingArguments(
      output_dir=save_name,
      num_train_epochs=3,
      per_device_train_batch_size=8,
      learning_rate=5e-7,
      warmup_ratio=0.1,
      weight_decay=0.01,
      fp16=False,
      bf16=False,
      max_grad_norm=0.1,
      logging_steps=10,
      save_strategy="epoch",
      report_to="none"
  )

    trainer=Trainer(
        model=model,
        args=args,
        train_dataset=ds,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer)
    )

    trainer.train()

    for name,param in model.named_parameters():
        if torch.isnan(param).any():
            print("NaN:",name)

    trainer.save_model(save_name)
    tokenizer.save_pretrained(save_name)

train_model("microsoft/deberta-v3-small","deberta_finetuned","deberta")
train_model("roberta-base","roberta_finetuned","roberta")

print("Training complete")

Device: cuda


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifie

Step,Training Loss
10,1.618880
20,1.631963
30,1.611362
40,1.646979
50,1.609401
60,1.612271
70,1.611322
80,1.608396
90,1.615407
100,1.608059


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete


Load the fine-tuned DeBERTa and RoBERTa models.

For the prompt at row index 25, perform inference using each model independently and apply Softmax to obtain class probabilities.

Question 1:

Which answer option receives the highest probability from the DeBERTa model, and what is that probability?

(answer format : eg - A, probability of A)

In [38]:
device="cuda" if torch.cuda.is_available() else "cpu"

deberta_tokenizer=AutoTokenizer.from_pretrained("deberta_finetuned")
deberta_model=AutoModelForSequenceClassification.from_pretrained("deberta_finetuned").to(device)
deberta_model.eval()

roberta_tokenizer=AutoTokenizer.from_pretrained("roberta_finetuned")
roberta_model=AutoModelForSequenceClassification.from_pretrained("roberta_finetuned").to(device)
roberta_model.eval()

print("Models loaded on",device)

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Models loaded on cuda


In [39]:
text=train_df.iloc[25]["prompt"]

inputs=deberta_tokenizer(text,return_tensors="pt",truncation=True,max_length=256).to(device)

with torch.no_grad():
    logits=deberta_model(**inputs).logits

deberta_probs=softmax(logits,dim=1)[0].cpu().numpy()

options=["A","B","C","D","E"]

print(options[deberta_probs.argmax()],deberta_probs.max())

A 0.4175


Using the same sample (row index 25), average the class probabilities from both models.

Average Probability = [P(DeBERTa) + P(RoBERTa)]/2

Question 2:

Which answer option receives the highest averaged probability after simple probability ensembling?

In [42]:
text=train_df.iloc[25]["prompt"]

inputs=roberta_tokenizer(text,return_tensors="pt",truncation=True,max_length=256).to(device)

with torch.no_grad():
    logits=roberta_model(**inputs).logits

roberta_probs=softmax(logits,dim=1)[0].cpu().numpy()

print(dict(zip(["A","B","C","D","E"],roberta_probs)))

{'A': np.float32(0.17871574), 'B': np.float32(0.23801537), 'C': np.float32(0.19032714), 'D': np.float32(0.19899571), 'E': np.float32(0.19394599)}


In [43]:
avg_probs=(deberta_probs+roberta_probs)/2

options=["A","B","C","D","E"]

print(dict(zip(options,avg_probs)))

answer=options[np.argmax(avg_probs)]
prob=avg_probs.max()

print(answer,prob)

{'A': np.float32(0.2980981), 'B': np.float32(0.122439004), 'C': np.float32(0.10032867), 'D': np.float32(0.28577715), 'E': np.float32(0.19340855)}
A 0.2980981


Apply weighted probability averaging after Softmax using the following weights:

DeBERTa: 0.70

RoBERTa: 0.30

Compute:

P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]

Question 3:

Which answer option is ranked first after weighted ensembling?

In [44]:
weighted_probs=(0.7*deberta_probs)+(0.3*roberta_probs)

options=["A","B","C","D","E"]

print(dict(zip(options,weighted_probs)))

print(options[np.argmax(weighted_probs)],weighted_probs.max())

{'A': np.float32(0.34585106), 'B': np.float32(0.07621113), 'C': np.float32(0.06433081), 'D': np.float32(0.3204409), 'E': np.float32(0.19319357)}
A 0.34585106


Using the weighted ensemble probabilities from Q3, rank all five answer options.

Write the final prediction exactly in Kaggle submission format.

Question 4:

What is the Top-3 prediction string for row index 25?

Example : C A E

In [45]:
options=["A","B","C","D","E"]

ranked_options=[options[i] for i in np.argsort(weighted_probs)[::-1]]

top3=" ".join(ranked_options[:3])

print(ranked_options)
print(top3)

['A', 'D', 'E', 'B', 'C']
A D E


Run the weighted ensemble pipeline on every row of test.csv.

Save the predictions in a file named submission.csv using the required Kaggle format:

id,prediction

where the prediction column contains the Top-3 ranked options separated by spaces.

Question 5:

Exactly how many prediction rows are present in the generated file (excluding the header)?

In [46]:
test_df=pd.read_csv("test.csv")
options=["A","B","C","D","E"]

predictions=[]

for text in test_df["prompt"]:
    inputs_d=deberta_tokenizer(text,return_tensors="pt",truncation=True,max_length=256).to(device)
    inputs_r=roberta_tokenizer(text,return_tensors="pt",truncation=True,max_length=256).to(device)

    with torch.no_grad():
        d_probs=softmax(deberta_model(**inputs_d).logits,dim=1)[0].cpu().numpy()
        r_probs=softmax(roberta_model(**inputs_r).logits,dim=1)[0].cpu().numpy()

    final_probs=(0.7*d_probs)+(0.3*r_probs)

    ranked=[options[i] for i in np.argsort(final_probs)[::-1]]
    predictions.append(" ".join(ranked[:3]))

submission=pd.DataFrame({
    "id":test_df["id"],
    "prediction":predictions
})

submission.to_csv("submission.csv",index=False)

print("Prediction rows:",len(submission))

Prediction rows: 500


For the first 50 rows of test.csv, create two versions of every prompt:

1.Original prompt

2.Instruction-augmented prompt by prepending: "Answer the following multiple-choice question carefully:"

Run inference using DeBERTa on both versions.

Average the predicted probabilities from both passes.

Question 6:

How many of the first 50 rows produce a different Top-1 prediction after applying Test-Time Augmentation?

In [47]:
options=["A","B","C","D","E"]

changes=0

for i in range(50):
    original=train_df.iloc[i]["prompt"]
    augmented="Answer the following multiple-choice question carefully: "+original

    probs=[]

    for text in [original,augmented]:
        inputs=deberta_tokenizer(text,return_tensors="pt",truncation=True,max_length=256).to(device)

        with torch.no_grad():
            logits=deberta_model(**inputs).logits

        probs.append(softmax(logits,dim=1)[0].cpu().numpy())

    avg_probs=(probs[0]+probs[1])/2

    original_top=options[np.argmax(probs[0])]
    tta_top=options[np.argmax(avg_probs)]

    if original_top!=tta_top:
        changes+=1

print(changes)

1


Process the first 100 rows of test.csv. And compare the Top-1 prediction from:

1. DeBERTa

2. Weighted Ensemble

Question 7:

How many rows have different Top-1 predictions?

In [48]:
options=["A","B","C","D","E"]

different=0

for i in range(100):
    text=train_df.iloc[i]["prompt"]

    d_inputs=deberta_tokenizer(text,return_tensors="pt",truncation=True,max_length=256).to(device)
    r_inputs=roberta_tokenizer(text,return_tensors="pt",truncation=True,max_length=256).to(device)

    with torch.no_grad():
        d_probs=softmax(deberta_model(**d_inputs).logits,dim=1)[0].cpu().numpy()
        r_probs=softmax(roberta_model(**r_inputs).logits,dim=1)[0].cpu().numpy()

    ensemble_probs=(0.7*d_probs)+(0.3*r_probs)

    deberta_top=options[np.argmax(d_probs)]
    ensemble_top=options[np.argmax(ensemble_probs)]

    if deberta_top!=ensemble_top:
        different+=1

print(different)

5


For the first 100 rows of test.csv, record the highest class probability (confidence) predicted by:

1. DeBERTa

2. Weighted Ensemble

For every row, compute:

Confidence Gain = Ensemble Confidence−DeBERTa Confidence

Question 8:

How many rows have a positive confidence gain (greater than 0)?

In [49]:
positive_gain=0

for i in range(100):
    text=train_df.iloc[i]["prompt"]

    d_inputs=deberta_tokenizer(text,return_tensors="pt",truncation=True,max_length=256).to(device)
    r_inputs=roberta_tokenizer(text,return_tensors="pt",truncation=True,max_length=256).to(device)

    with torch.no_grad():
        d_probs=softmax(deberta_model(**d_inputs).logits,dim=1)[0].cpu().numpy()
        r_probs=softmax(roberta_model(**r_inputs).logits,dim=1)[0].cpu().numpy()

    ensemble_probs=(0.7*d_probs)+(0.3*r_probs)

    deberta_conf=np.max(d_probs)
    ensemble_conf=np.max(ensemble_probs)

    confidence_gain=ensemble_conf-deberta_conf

    if confidence_gain>0:
        positive_gain+=1

print(positive_gain)

0


For the first 100 rows of test.csv, compare the Top-3 prediction strings generated by:

1. DeBERTa alone

2. Weighted Ensemble

Question 9:

How many rows have at least one change in their ordered Top-3 ranking after ensembling?

Examples:

A C D vs. A D C

In [50]:
options=["A","B","C","D","E"]

changes=0

for i in range(100):
    text=train_df.iloc[i]["prompt"]

    d_inputs=deberta_tokenizer(text,return_tensors="pt",truncation=True,max_length=256).to(device)
    r_inputs=roberta_tokenizer(text,return_tensors="pt",truncation=True,max_length=256).to(device)

    with torch.no_grad():
        d_probs=softmax(deberta_model(**d_inputs).logits,dim=1)[0].cpu().numpy()
        r_probs=softmax(roberta_model(**r_inputs).logits,dim=1)[0].cpu().numpy()

    ensemble_probs=(0.7*d_probs)+(0.3*r_probs)

    deberta_top3=" ".join([options[j] for j in np.argsort(d_probs)[::-1][:3]])
    ensemble_top3=" ".join([options[j] for j in np.argsort(ensemble_probs)[::-1][:3]])

    if deberta_top3!=ensemble_top3:
        changes+=1

print(changes)

10


Using the Top-3 predictions generated by your weighted ensemble for the first 100 validation samples, compute the MAP@3 score.

Question 10:


What is the final MAP@3 score?

(Round to 4 decimal places.)

In [51]:
options=["A","B","C","D","E"]

scores=[]

for i in range(100):
    text=train_df.iloc[i]["prompt"]

    true_answer=train_df.iloc[i]["answer"]
    true_answer=str(true_answer).strip().upper()[0]

    d_inputs=deberta_tokenizer(text,return_tensors="pt",truncation=True,max_length=256).to(device)
    r_inputs=roberta_tokenizer(text,return_tensors="pt",truncation=True,max_length=256).to(device)

    with torch.no_grad():
        d_probs=softmax(deberta_model(**d_inputs).logits,dim=1)[0].cpu().numpy()
        r_probs=softmax(roberta_model(**r_inputs).logits,dim=1)[0].cpu().numpy()

    ensemble_probs=(0.7*d_probs)+(0.3*r_probs)

    top3=[options[j] for j in np.argsort(ensemble_probs)[::-1][:3]]

    if true_answer in top3:
        rank=top3.index(true_answer)+1
        scores.append(1/rank)
    else:
        scores.append(0)

map3=np.mean(scores)

print(round(map3,4))

0.7583
